# 01 · NetFlow v3 data stage (NF-UNSW-NB15-v3 and NF-ToN-IoT-v3)

Design decisions:
- Source: NetFlow v3 datasets from the University of Queensland (Luay et al. 2025, arXiv 2503.04404), 53 NetFlow features.
- Raw archives stay in the Drive project folder, outside the repository; processed parquet files go to `XIDS_Certify_Research/data/nf3_v1/`.
- Model inputs: 46 features. Excluded as provenance: IPv4 source and destination addresses, source and destination ports, flow start and end timestamps. `DNS_QUERY_ID` is also excluded, because it is a random transaction identifier.
- Label: the binary `Label` column. Rows whose `Label` contradicts the `Attack` column are quarantined.
- Rows with a missing or non-finite model feature are dropped and counted per class.
- Exact model-feature duplicates collapse to one row with a multiplicity count; duplicate groups that disagree on the label are quarantined.
- Clusters: conversation-minute groups (source address, destination address, destination port, protocol, 60 s window of the flow start). A cluster never crosses roles.
- Four disjoint roles: detector_train 0.50, development 0.15, calibration 0.15, locked_test 0.20, stratified by attack family.
- Each CSV is read twice in 1M-row chunks, so memory stays bounded for the 27.5M-row ToN-IoT file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import gc, importlib, json, os, sys
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/XIDS_Certify_Research')
REPO = PROJECT / 'xids-certify'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
assert (REPO / 'tools' / 'commit_cell.py').exists(), 'tools/commit_cell.py is missing from the repo'

import numpy as np
import pandas as pd
import src.nf3_data as nf
importlib.reload(nf)

WORK = Path('/content/nf3_work')             # runtime disk: fast reads for the large archives
TABLES = REPO / 'results' / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)
OUT = PROJECT / 'data' / 'nf3_v1'
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
print(f'project: {PROJECT} | pandas {pd.__version__} | numpy {np.__version__} | model features: {len(nf.MODEL_FEATURES)}')

## Archives: identify each file from the CSV inside it

In [ ]:
archives = nf.find_archives(PROJECT, REPO)
display(archives[['file', 'member', 'dataset', 'bytes', 'member_bytes']])
found = archives.dropna(subset=['dataset'])
missing = {'unsw', 'ton'} - set(found['dataset'])
assert not missing, f'no archive found for: {sorted(missing)}'
assert not found['dataset'].duplicated().any(), 'two archives hold the same dataset'
SRC = {r.dataset: (Path(r.path), r.member) for r in found.itertuples()}
manifest = []

## NF-UNSW-NB15-v3: schema, labels, duplicates, clusters and roles

In [ ]:
ds = 'unsw'
local = nf.stage_local(SRC[ds][0], WORK)
manifest.append({'dataset': ds, 'file': SRC[ds][0].name, 'member': SRC[ds][1],
                 'bytes': local.stat().st_size, 'sha256': nf.sha256_file(local)})
arrays, fams, schema_u, info_u = nf.scan(local, SRC[ds][1], ds)
plan_u, audit_u, role_u, fam_u, labels_u = nf.decide(arrays, fams, ds)
print(info_u)
display(labels_u)
display(pd.Series(audit_u).to_frame('value'))
display(role_u)
display(fam_u)
display(schema_u[schema_u['constant'] | (schema_u['n_missing'] > 0)])

In [ ]:
nf.write_parquet(local, SRC[ds][1], plan_u, fams, ds, OUT / ds)
check = nf.load_split(PROJECT, ds, columns=['cluster_id', 'role'])
assert len(check) == audit_u['rows_after_collapse']
assert check.groupby('cluster_id')['role'].nunique().max() == 1
print(f"{ds}: {len(check):,} rows written to {OUT / ds / 'flows.parquet'}")
del arrays, plan_u, check
local.unlink()
gc.collect()

## NF-ToN-IoT-v3: schema, labels, duplicates, clusters and roles

In [ ]:
ds = 'ton'
local = nf.stage_local(SRC[ds][0], WORK)
manifest.append({'dataset': ds, 'file': SRC[ds][0].name, 'member': SRC[ds][1],
                 'bytes': local.stat().st_size, 'sha256': nf.sha256_file(local)})
arrays, fams, schema_t, info_t = nf.scan(local, SRC[ds][1], ds)
plan_t, audit_t, role_t, fam_t, labels_t = nf.decide(arrays, fams, ds)
print(info_t)
display(labels_t)
display(pd.Series(audit_t).to_frame('value'))
display(role_t)
display(fam_t)
display(schema_t[schema_t['constant'] | (schema_t['n_missing'] > 0)])

In [ ]:
nf.write_parquet(local, SRC[ds][1], plan_t, fams, ds, OUT / ds)
check = nf.load_split(PROJECT, ds, columns=['cluster_id', 'role'])
assert len(check) == audit_t['rows_after_collapse']
assert check.groupby('cluster_id')['role'].nunique().max() == 1
print(f"{ds}: {len(check):,} rows written to {OUT / ds / 'flows.parquet'}")
del arrays, plan_t, check
local.unlink()
gc.collect()

## Tables, split hashes and configuration

In [ ]:
pd.DataFrame(manifest).to_csv(TABLES / 'nf01_archives.csv', index=False)
pd.concat([schema_u, schema_t], ignore_index=True).to_csv(TABLES / 'nf01_schema.csv', index=False)
pd.concat([labels_u, labels_t], ignore_index=True).to_csv(TABLES / 'nf01_labels.csv', index=False)
pd.concat([role_u, role_t], ignore_index=True).to_csv(TABLES / 'nf01_split_by_role.csv', index=False)
pd.concat([fam_u, fam_t], ignore_index=True).to_csv(TABLES / 'nf01_split_by_family.csv', index=False)
audit = pd.DataFrame([{**info_u, **audit_u}, {**info_t, **audit_t}]).set_index('dataset')
audit.to_csv(TABLES / 'nf01_split_audit.csv')
config_hash = nf.write_config(REPO)
summary = {'config': 'configs/data_nf3_v1.json', 'config_sha256': config_hash,
           'rows_read': {'unsw': info_u['rows_read'], 'ton': info_t['rows_read']},
           'rows_published': {'unsw': info_u['rows_published'], 'ton': info_t['rows_published']},
           'rows_after_collapse': {'unsw': audit_u['rows_after_collapse'], 'ton': audit_t['rows_after_collapse']},
           'split_hash': {'unsw': audit_u['split_hash'], 'ton': audit_t['split_hash']},
           'model_features': len(nf.MODEL_FEATURES)}
(TABLES / 'nf01_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
display(audit.T)
print(json.dumps(summary, indent=2))

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/XIDS_Certify_Research/xids-certify")
msg = "Notebook 01: NetFlow v3 data stage for NF-UNSW-NB15-v3 and NF-ToN-IoT-v3. Archive identification and hashes, schema check against the published 53 features, binary label policy with Label/Attack contradictions quarantined, duplicate collapse with label-conflict quarantine, conversation-minute clusters, four disjoint roles with split hashes"
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)